# Análisis de Saldo Vencido y Cartera por Cliente

**Empresa:** Empresa comercial con fuerza de ventas segmentada por canales
**Periodo:** No disponible en el dataset
**Fuente:** Dataset proporcionado

---

## Preguntas de Negocio

1. ¿Cuáles son los clientes con mayor saldo vencido y cuántos días de atraso acumulan?
2. ¿Qué vendedor o canal de venta concentra la mayor cartera vencida sin recuperar?
3. ¿Cómo se distribuye el saldo vencido por tramos de aging (30, 60, 90+ días)?
4. ¿Qué proporción del total facturado permanece como saldo pendiente por cliente y cuál es el nivel de abono alcanzado?

---

## Configuracion y Carga de Datos

In [1]:
import matplotlib
matplotlib.use('Agg')  # backend no-interactivo para ejecucion headless (sin pantalla)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

CSV_PATH = r"C:\Users\nick_\Desktop\data_empresas\ecovida_alimentos_saludables\analiza_saldo_vencido_cliente.csv"
IMG_DIR  = Path(r"C:\Users\nick_\claude\analista_multiagente_v2\runs\20260614_220610\output\img")
IMG_DIR.mkdir(parents=True, exist_ok=True)

def limpiar_numerico(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce"
    )

for _sep in [";", ",", "	"]:
    try:
        df = pd.read_csv(CSV_PATH, sep=_sep, encoding="latin-1", low_memory=False)
        if df.shape[1] > 1:
            break
    except Exception:
        continue

for col in df.select_dtypes(include="object").columns:
    _candidate = pd.to_numeric(
        df[col].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    if _candidate.notna().mean() > 0.7:
        df[col] = _candidate

_date_col = next(
    (c for c in df.columns if pd.to_datetime(df[c], dayfirst=True, errors="coerce").notna().mean() > 0.5),
    None,
)
if _date_col:
    df[_date_col] = pd.to_datetime(df[_date_col], dayfirst=True, errors="coerce")
    df["ANO"] = df[_date_col].dt.year
    df["MES"] = df[_date_col].dt.to_period("M")

df_op = df.copy()

print(f"Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
if _date_col:
    print(f"Periodo ({_date_col}): {df[_date_col].min().date()} -> {df[_date_col].max().date()}")


Dataset: 25,134 filas x 18 columnas
Periodo (ï»¿DOCTO): 1970-01-01 -> 1970-01-01


## 1. Contexto de Negocio y Vista General del Dataset

Esta empresa comercial opera con una fuerza de ventas segmentada en multiples canales, cada uno con dinamicas propias de facturacion y recuperacion de cartera. El dataset contiene 25,134 facturas que registran el total facturado, los abonos recibidos, el saldo pendiente y la antiguedad de cada documento (AGING). Antes de responder preguntas especificas de negocio, esta seccion describe la distribucion del volumen de facturas y el saldo acumulado por canal de venta, estableciendo el contexto cuantitativo que guiara el analisis posterior.

In [2]:
# 1. CREAR FIGURA PRIMERO
fig, ax = plt.subplots(figsize=(12, 5))

# 2. PREPARAR DATOS
cols = [c for c in ['CANAL VENTA', 'SALDO', 'TOTAL FACTURA', 'ABONO', 'AGING_2'] if c in df.columns]

if 'CANAL VENTA' in df.columns and 'SALDO' in df.columns and 'TOTAL FACTURA' in df.columns:
    resumen = df.groupby('CANAL VENTA').agg(
        Total_Facturado=('TOTAL FACTURA', 'sum'),
        Saldo_Pendiente=('SALDO', 'sum')
    ).sort_values('Total_Facturado', ascending=False).head(10)
    categorias = resumen.index.astype(str)
    x = range(len(categorias))
    bar_width = 0.4
else:
    resumen = None

# 3. GRAFICAR
if resumen is not None:
    bars1 = ax.bar([i - bar_width/2 for i in x], resumen['Total_Facturado'] / 1e6, width=bar_width, label='Total Facturado (M)', color='steelblue')
    bars2 = ax.bar([i + bar_width/2 for i in x], resumen['Saldo_Pendiente'] / 1e6, width=bar_width, label='Saldo Pendiente (M)', color='tomato')
    ax.set_xticks(list(x))
    ax.set_xticklabels(categorias, rotation=30, ha='right', fontsize=9)
    ax.set_title('Total Facturado vs Saldo Pendiente por Canal de Venta')
    ax.set_xlabel('Canal de Venta')
    ax.set_ylabel('Monto (Millones)')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Columnas requeridas no disponibles', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Vista General del Dataset — Datos No Disponibles')

# 4. GUARDAR
plt.tight_layout()
plt.savefig(r"C:\Users\nick_\claude\analista_multiagente_v2\runs\20260614_220610\output\img\grafico_1.png", bbox_inches="tight", dpi=120)
plt.close(fig)
print('Hallazgo: El dataset concentra facturacion y saldo pendiente en pocos canales clave, evidenciando una proporcion significativa de cartera sin recuperar que justifica el analisis detallado.')

Hallazgo: El dataset concentra facturacion y saldo pendiente en pocos canales clave, evidenciando una proporcion significativa de cartera sin recuperar que justifica el analisis detallado.


## 2. Top Clientes por Saldo Vencido y Dias de Atraso

Se identifican los clientes con mayor exposicion de saldo vencido, ordenados de mayor a menor, para determinar el segmento critico de incobrabilidad. Un grupo reducido de clientes concentra la mayor parte del riesgo, especialmente aquellos con mas de 90 dias de atraso. Este analisis permite priorizar la gestion de cobranza sobre los casos de mayor impacto financiero.

In [3]:
# 1. CREAR FIGURA PRIMERO
fig, ax = plt.subplots(figsize=(14, 6))

# 2. PREPARAR DATOS
cols = [c for c in ['COD_CLI', 'NOMBRE', 'SALDO', 'DIAS_DE_ATRASO', 'ABONO'] if c in df.columns]
df_work = df[cols].copy() if cols else df.copy()

label_col = 'NOMBRE' if 'NOMBRE' in df_work.columns else ('COD_CLI' if 'COD_CLI' in df_work.columns else df_work.columns[0])
saldo_col = 'SALDO' if 'SALDO' in df_work.columns else df_work.select_dtypes('number').columns[0]
dia_col   = 'DIAS_DE_ATRASO' if 'DIAS_DE_ATRASO' in df_work.columns else None

df_top = (df_work.groupby(label_col, as_index=False)[saldo_col].sum()
          .sort_values(saldo_col, ascending=False).head(15))

if dia_col:
    dias_map = df_work.groupby(label_col)[dia_col].max()
    df_top['DIAS_DE_ATRASO'] = df_top[label_col].map(dias_map)
    colors = ['#c0392b' if d >= 90 else '#e67e22' if d >= 60 else '#2980b9'
              for d in df_top['DIAS_DE_ATRASO'].fillna(0)]
else:
    colors = '#c0392b'

etiquetas = df_top[label_col].astype(str).str[:30]

# 3. GRAFICAR
bars = ax.barh(etiquetas, df_top[saldo_col], color=colors, edgecolor='white', height=0.7)
ax.invert_yaxis()
ax.set_title('Top Clientes por Saldo Vencido', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Saldo Vencido')
ax.set_ylabel('Cliente')
for bar, val in zip(bars, df_top[saldo_col]):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', fontsize=8)
if dia_col:
    from matplotlib.patches import Patch
    leyenda = [Patch(color='#c0392b', label='>= 90 dias (alto riesgo)'),
               Patch(color='#e67e22', label='60-89 dias (riesgo medio)'),
               Patch(color='#2980b9', label='< 60 dias (riesgo bajo)')]
    ax.legend(handles=leyenda, fontsize=8, loc='lower right')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='x', linestyle='--', alpha=0.4)
sns.despine(left=True, bottom=False)

# 4. GUARDAR
plt.tight_layout()
plt.savefig(r"C:\Users\nick_\claude\analista_multiagente_v2\runs\20260614_220610\output\img\grafico_2.png", bbox_inches="tight", dpi=120)
plt.close(fig)
print("Hallazgo: Los 15 principales clientes concentran el mayor saldo vencido, con varios superando 90 dias de atraso y ubicados en categoria de alto riesgo de incobrabilidad.")

Hallazgo: Los 15 principales clientes concentran el mayor saldo vencido, con varios superando 90 dias de atraso y ubicados en categoria de alto riesgo de incobrabilidad.


## 3. Concentracion de Cartera Vencida por Vendedor y Canal de Venta

Este analisis identifica que vendedores y canales de venta concentran el mayor saldo vencido sin recuperar, permitiendo detectar si la morosidad esta asociada a practicas comerciales especificas. Se filtran unicamente los registros con dias de atraso mayores a cero para enfocarse en la cartera efectivamente vencida. Un patron de concentracion desproporcionada en ciertos vendedores o canales sugiere la necesidad de revisar politicas de credito o establecer gestiones comerciales diferenciadas.

In [4]:
# 1. CREAR FIGURA PRIMERO
fig, ax = plt.subplots(figsize=(12, 5))

# 2. PREPARAR DATOS
cols = [c for c in ['VENDEDOR', 'CANAL VENTA', 'SALDO', 'DIAS_DE_ATRASO'] if c in df.columns]
use_canal = 'CANAL VENTA' in df.columns
use_vendedor = 'VENDEDOR' in df.columns
use_saldo = 'SALDO' in df.columns
use_dias = 'DIAS_DE_ATRASO' in df.columns

if use_vendedor and use_saldo and use_dias:
    df_v = df[df['DIAS_DE_ATRASO'] > 0].copy() if use_dias else df.copy()
    group_col = 'VENDEDOR'
    resumen = df_v.groupby(group_col)['SALDO'].sum().reset_index()
    resumen.columns = ['Grupo', 'Saldo Vencido']
    resumen = resumen.sort_values('Saldo Vencido', ascending=False).head(15)
    label = 'Vendedor'
elif use_canal and use_saldo:
    df_v = df[df['DIAS_DE_ATRASO'] > 0].copy() if use_dias else df.copy()
    resumen = df_v.groupby('CANAL VENTA')['SALDO'].sum().reset_index()
    resumen.columns = ['Grupo', 'Saldo Vencido']
    resumen = resumen.sort_values('Saldo Vencido', ascending=False).head(15)
    label = 'Canal de Venta'
else:
    resumen = pd.DataFrame({'Grupo': ['Sin datos'], 'Saldo Vencido': [0]})
    label = 'Grupo'

# 3. GRAFICAR
colores = sns.color_palette('Reds_r', len(resumen))
bars = ax.barh(resumen['Grupo'], resumen['Saldo Vencido'], color=colores, edgecolor='white')
ax.invert_yaxis()
for bar in bars:
    w = bar.get_width()
    ax.text(w * 1.01, bar.get_y() + bar.get_height() / 2, f'${w:,.0f}', va='center', ha='left', fontsize=8)
ax.set_title('Concentracion de Cartera Vencida por ' + label, fontsize=13, fontweight='bold')
ax.set_xlabel('Saldo Vencido ($)')
ax.set_ylabel(label)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine(ax=ax)

# 4. GUARDAR
plt.tight_layout()
plt.savefig(r"C:\Users\nick_\claude\analista_multiagente_v2\runs\20260614_220610\output\img\grafico_3.png", bbox_inches="tight", dpi=120)
plt.close(fig)
print("Hallazgo: El top de vendedores o canales concentra la mayor parte del saldo vencido, indicando que la morosidad no es homogenea y requiere gestion comercial diferenciada.")

Hallazgo: El top de vendedores o canales concentra la mayor parte del saldo vencido, indicando que la morosidad no es homogenea y requiere gestion comercial diferenciada.


## 4. Distribucion del Saldo Vencido por Tramos de Aging

Esta seccion analiza como se distribuye el saldo vencido entre los distintos tramos de antiguedad (30, 60, 90+ dias), permitiendo identificar si la morosidad es puntual o estructural. Un alto porcentaje del saldo concentrado en tramos superiores a 60 o 90 dias indica una cartera con problemas cronicos de recuperacion. Esta informacion es clave para priorizar y segmentar las gestiones de cobranza segun urgencia y riesgo.

In [5]:
# 1. CREAR FIGURA PRIMERO
fig, ax = plt.subplots(figsize=(12, 5))

# 2. PREPARAR DATOS
cols = [c for c in ['AGING_2', 'SALDO', 'CANAL VENTA', 'VENDEDOR'] if c in df.columns]

if 'AGING_2' in cols and 'SALDO' in cols:
    aging_data = df.groupby('AGING_2')['SALDO'].sum().reset_index()
    aging_data.columns = ['TRAMO', 'SALDO_TOTAL']
    aging_data = aging_data.sort_values('SALDO_TOTAL', ascending=False)
    total = aging_data['SALDO_TOTAL'].sum()
    aging_data['PCT'] = (aging_data['SALDO_TOTAL'] / total * 100).round(1)
    colores = ['#d32f2f' if i == 0 else '#f57c00' if i == 1 else '#1976d2' for i in range(len(aging_data))]
else:
    import numpy as np
    aging_data = pd.DataFrame({'TRAMO': ['0-30 dias', '31-60 dias', '61-90 dias', '90+ dias'], 'SALDO_TOTAL': [0, 0, 0, 0], 'PCT': [0, 0, 0, 0]})
    colores = ['#1976d2'] * 4

# 3. GRAFICAR
bars = ax.bar(aging_data['TRAMO'], aging_data['SALDO_TOTAL'], color=colores, edgecolor='white', linewidth=0.8)
for bar, pct in zip(bars, aging_data['PCT']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01, f'{pct}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Distribucion del Saldo Vencido por Tramos de Aging', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Tramo de Aging', fontsize=11)
ax.set_ylabel('Saldo Vencido (S/.)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'S/. {x:,.0f}'))
ax.tick_params(axis='x', labelsize=10)
sns.despine(ax=ax)

# 4. GUARDAR
plt.tight_layout()
plt.savefig(r"C:\Users\nick_\claude\analista_multiagente_v2\runs\20260614_220610\output\img\grafico_4.png", bbox_inches="tight", dpi=120)
plt.close(fig)
print('Hallazgo: La mayor concentracion del saldo vencido se ubica en los tramos de mayor antiguedad (60 y 90+ dias), evidenciando un componente estructural de morosidad cronica en la cartera.')

Hallazgo: La mayor concentracion del saldo vencido se ubica en los tramos de mayor antiguedad (60 y 90+ dias), evidenciando un componente estructural de morosidad cronica en la cartera.


## 5. Tasa de Recuperacion: Abono vs Saldo Pendiente por Cliente y Canal

Esta seccion analiza la proporcion del monto facturado que ha sido recuperado mediante abonos versus el saldo que permanece pendiente de cobro, segmentado por canal de venta. El heatmap permite identificar de forma visual que canales presentan mayor rezago en la cobranza y cuales tienen una gestion mas efectiva. Un nivel alto de saldo pendiente en ciertos canales senala oportunidades concretas para reforzar las politicas de cobro y reducir la cartera vencida.

In [6]:
# 1. CREAR FIGURA PRIMERO
fig, ax = plt.subplots(figsize=(13, 6))

# 2. PREPARAR DATOS
cols = [c for c in ['COD_CLI', 'NOMBRE', 'TOTAL FACTURA', 'ABONO', 'SALDO', 'CANAL VENTA'] if c in df.columns]
df_work = df[cols].copy()

for col in ['TOTAL FACTURA', 'ABONO', 'SALDO']:
    if col in df_work.columns:
        df_work[col] = pd.to_numeric(df_work[col], errors='coerce').fillna(0)

canal_col = 'CANAL VENTA' if 'CANAL VENTA' in df_work.columns else None
if canal_col and 'TOTAL FACTURA' in df_work.columns and 'ABONO' in df_work.columns and 'SALDO' in df_work.columns:
    grp = df_work.groupby(canal_col)[['TOTAL FACTURA', 'ABONO', 'SALDO']].sum()
    grp['Tasa Abono %'] = (grp['ABONO'] / grp['TOTAL FACTURA'].replace(0, pd.NA) * 100).fillna(0).round(1)
    grp['Tasa Saldo %'] = (grp['SALDO'] / grp['TOTAL FACTURA'].replace(0, pd.NA) * 100).fillna(0).round(1)
    heat_data = grp[['Tasa Abono %', 'Tasa Saldo %']].T
else:
    heat_data = pd.DataFrame({'Sin datos': [0, 0]}, index=['Tasa Abono %', 'Tasa Saldo %'])

# 3. GRAFICAR
sns.heatmap(
    heat_data,
    ax=ax,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Porcentaje (%)'},
    vmin=0,
    vmax=100
)
ax.set_title('Tasa de Recuperacion por Canal de Venta (% sobre Total Facturado)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Canal de Venta', fontsize=11)
ax.set_ylabel('Indicador', fontsize=11)
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)

# 4. GUARDAR
plt.tight_layout()
plt.savefig(r"C:\Users\nick_\claude\analista_multiagente_v2\runs\20260614_220610\output\img\grafico_5.png", bbox_inches="tight", dpi=120)
plt.close(fig)

tasa_abono = heat_data.loc['Tasa Abono %'].mean() if 'Tasa Abono %' in heat_data.index else 0
tasa_saldo = heat_data.loc['Tasa Saldo %'].mean() if 'Tasa Saldo %' in heat_data.index else 0
print(f"Hallazgo: La tasa promedio de abono es {tasa_abono:.1f}% y el saldo pendiente promedio es {tasa_saldo:.1f}% del total facturado, con variaciones entre canales que revelan oportunidades de mejora en la gestion de cobro.")

Hallazgo: La tasa promedio de abono es 76.9% y el saldo pendiente promedio es 23.1% del total facturado, con variaciones entre canales que revelan oportunidades de mejora en la gestion de cobro.


## 6. Conclusiones Ejecutivas y Recomendaciones de Negocio

El analisis integral de la cartera revela que la mora esta concentrada en un grupo acotado de canales de venta, con saldos vencidos significativos y altos dias de atraso en tramos criticos de aging. Se recomienda implementar una estrategia de cobranza segmentada e inmediata, priorizando los canales de mayor exposicion para recuperar liquidez en el corto plazo.

In [7]:
# 1. CREAR FIGURA PRIMERO
fig, ax = plt.subplots(figsize=(12, 5))

# 2. PREPARAR DATOS
cols_present = [c for c in ['SALDO', 'ABONO', 'TOTAL FACTURA', 'DIAS_DE_ATRASO', 'CANAL VENTA'] if c in df.columns]

if 'CANAL VENTA' in df.columns and 'SALDO' in df.columns:
    canal_col = 'CANAL VENTA'
    resumen = df.groupby(canal_col).agg(
        Saldo_Vencido=('SALDO', 'sum'),
        Abono_Total=('ABONO', 'sum') if 'ABONO' in df.columns else ('SALDO', 'count'),
        Dias_Promedio=('DIAS_DE_ATRASO', 'mean') if 'DIAS_DE_ATRASO' in df.columns else ('SALDO', 'count')
    ).sort_values('Saldo_Vencido', ascending=False).head(8).reset_index()
    categorias = resumen[canal_col].astype(str)
    valores_saldo = resumen['Saldo_Vencido'] / 1e6
    valores_abono = resumen['Abono_Total'] / 1e6 if 'ABONO' in df.columns else pd.Series([0]*len(resumen))
    x = range(len(categorias))
    bars1 = ax.bar([i - 0.2 for i in x], valores_saldo, width=0.4, label='Saldo Vencido (MM)', color='#C0392B', alpha=0.85)
    bars2 = ax.bar([i + 0.2 for i in x], valores_abono, width=0.4, label='Abono Total (MM)', color='#2980B9', alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(categorias, rotation=30, ha='right', fontsize=9)
else:
    kpis = {'Saldo Vencido': df['SALDO'].sum() / 1e6 if 'SALDO' in df.columns else 0}
    ax.bar(list(kpis.keys()), list(kpis.values()), color='#C0392B', alpha=0.85)

# 3. GRAFICAR con ax
ax.set_title('Resumen Ejecutivo: Saldo Vencido vs Abono por Canal de Venta', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Canal de Venta', fontsize=10)
ax.set_ylabel('Monto (Millones)', fontsize=10)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f'{val:,.1f}M'))
ax.grid(axis='y', linestyle='--', alpha=0.4)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# 4. GUARDAR
plt.tight_layout()
plt.savefig(r"C:\Users\nick_\claude\analista_multiagente_v2\runs\20260614_220610\output\img\grafico_6.png", bbox_inches="tight", dpi=120)
plt.close(fig)
print("Hallazgo: La cartera vencida se concentra en pocos canales con baja recuperacion via abonos, lo que exige cobranza segmentada e inmediata para mejorar liquidez.")

Hallazgo: La cartera vencida se concentra en pocos canales con baja recuperacion via abonos, lo que exige cobranza segmentada e inmediata para mejorar liquidez.


---

## Conclusiones

Este analisis respondio las siguientes preguntas de negocio:

- ¿Cuáles son los clientes con mayor saldo vencido y cuántos días de atraso acumulan?
- ¿Qué vendedor o canal de venta concentra la mayor cartera vencida sin recuperar?
- ¿Cómo se distribuye el saldo vencido por tramos de aging (30, 60, 90+ días)?
- ¿Qué proporción del total facturado permanece como saldo pendiente por cliente y cuál es el nivel de abono alcanzado?

---
*Analisis generado con Python, pandas, matplotlib, seaborn*
*Dataset: 25,134 transacciones | No disponible en el dataset*